# Modular argument graph alignment and spectral partitioning

This notebook composes six reusable modules extracted from `1.7_graph_construction_node_attribute_fixed.ipynb`:

1. reading and validation;
2. node embeddings;
3. graph construction and relation-aware transitions;
4. cross-graph similarity;
5. global matrix `H` construction;
6. spectral partitioning.

The final compatibility checks verify the original notebook's dataset, matrix, affinity-graph, and partition results.

In [1]:
from __future__ import annotations

import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

search_roots = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    root for root in search_roots
    if (root / "src" / "legal_data_doc_parts").is_dir()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from legal_data_doc_parts.argument_graph import (
    ArgumentGraphBuilder,
    GlobalSimilarityMatrixBuilder,
    LocalSpectralPartitioner,
    PartitionConfig,
    RelationAwareIsoRank,
    TransformerMeanPoolEmbedder,
    embed_node_attributes,
    load_and_validate_documents,
    locate_project_resource,
    mutual_knn_graph,
    validate_and_restore_partition,
    validate_global_similarity,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("modular-argument-graph-pipeline")

RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBEDDING_BATCH_SIZE = 64
ISORANK_ALPHA = 0.15
ISORANK_TOL = 1e-5
ISORANK_MAX_ITER = 500
K_NEIGHBORS = 5
MIN_SIMILARITY = 0.55

PARTITION_CONFIG = PartitionConfig(
    restart=0.15,
    pagerank_tolerance=1e-10,
    pagerank_max_iterations=500,
    star_radius=2,
    max_star_nodes=128,
    maximum_conductance=0.80,
    one_node_per_graph=True,
)

logger.info("PyTorch device: %s", DEVICE)

2026-11-23 06:57:02,981 | INFO | PyTorch device: cuda


In [2]:
paragraph_dataset_mini = [
    "116062204", "116075685", "116080900", "116084352", "116084886",
    "116099338", "116102314", "116105977", "116106432", "116109598",
    "116110555", "116116081", "116118921", "116120654", "116121714",
    "116125191", "116126076", "116127831", "116141779", "116142769",
    "116149190", "116150283", "116167789", "116168634", "116170057",
    "116170822", "116192026", "116194637", "116196887", "116197578",
    "116199866", "116204676", "116213690", "116214232", "116220471",
    "116221948", "116226431", "116229939", "116232414", "116233794",
    "116245112", "116246991", "116248468", "116252607", "116253514",
    "116256069", "116257159", "116259036", "116260246", "116261015",
    "116261263", "116270112", "116277632", "116278949", "116283678",
    "116285648", "116289180", "116308208", "116310154", "116320626",
    "116321603", "116336690", "116339406", "116344031", "116350541",
    "116350825", "116358099", "116359089", "116376162", "116377931",
    "116385451", "116398308", "116410200", "116410462", "116414218",
    "116432495", "116434259", "116442566", "116462199", "116588653",
    "116681882", "116764649", "116780853", "116790074", "116794466",
    "116972204", "117069850", "117125134", "117130242", "117230819",
    "117251852", "117420480", "117440182", "117456818", "117460145",
    "117464049", "117489745", "117516936", "117556435",
]

assert paragraph_dataset_mini
assert len(paragraph_dataset_mini) == len(set(paragraph_dataset_mini))
logger.info("Configured %d documents", len(paragraph_dataset_mini))

2026-11-23 06:57:03,117 | INFO | Configured 99 documents


## 1. Reading and validation

The reader handles file access. The parser independently validates schema, references, confidence values, and global row ordering.

In [3]:
DATA_DIR = locate_project_resource(Path("document_data_parsing"))
ENCODER_DIR = locate_project_resource(
    Path("bge_model/bge-small-uk-law-v1_1/final/encoder")
)
TOKENIZER_DIR = locate_project_resource(
    Path("bge_model/bge-small-uk-law-v1_1/final/tokenizer")
)

dataset = load_and_validate_documents(paragraph_dataset_mini, DATA_DIR)
doc_args = dataset.documents
arguments_df = dataset.arguments
relations_df = dataset.relations

dataset_summary = pd.Series(
    {
        "documents": len(doc_args),
        "arguments": len(arguments_df),
        "relations": len(relations_df),
        "premises": int((arguments_df["argument_type"] == "PREMISE").sum()),
        "claims": int((arguments_df["argument_type"] == "CLAIM").sum()),
        "duplicate_source_target_pairs": int(
            relations_df.duplicated(["document_id", "source", "target"]).sum()
        ),
    },
    name="count",
)
display(dataset_summary.to_frame())

,count
documents,99
arguments,3941
relations,2928
premises,2710
claims,1231
duplicate_source_target_pairs,15


## 2. Node embeddings

The transformer adapter owns tokenization, batching, pooling, normalization, and device placement. The pipeline service aligns text and optional section embeddings with the validated argument rows.

In [4]:
embedder = TransformerMeanPoolEmbedder.from_pretrained(
    ENCODER_DIR,
    TOKENIZER_DIR,
    device=DEVICE,
    batch_size=EMBEDDING_BATCH_SIZE,
)
embedded_nodes = embed_node_attributes(
    arguments_df,
    embedder,
    device=DEVICE,
)
text_embeddings = embedded_nodes.text_embeddings
section_embeddings = embedded_nodes.section_embeddings
section_present = embedded_nodes.section_present

logger.info(
    "Encoded %d arguments into %d dimensions",
    text_embeddings.shape[0],
    text_embeddings.shape[1],
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

2026-11-23 06:57:14,919 | INFO | Encoded 3941 arguments into 384 dimensions


## 3. Graph construction and relation-aware transitions

The graph builder creates one `MultiDiGraph` per document. The transition builder keeps relation type and direction in separate channels, weights duplicate relations by confidence, and gives isolated nodes a stochastic self channel.

In [5]:
graph_collection = ArgumentGraphBuilder(device=DEVICE).build(
    arguments_df,
    relations_df,
    embedded_nodes,
    paragraph_dataset_mini,
)
case_data = graph_collection.cases
cases = graph_collection.graphs
graph_sizes = graph_collection.graph_sizes
offsets = graph_collection.offsets
channel_names = graph_collection.channel_names

graph_summary = pd.DataFrame(
    {
        "document_id": paragraph_dataset_mini,
        "nodes": graph_sizes,
        "relations": [graph.number_of_edges() for graph in cases],
        "isolated_nodes": [
            sum(graph.degree(node) == 0 for node in graph.nodes)
            for graph in cases
        ],
    }
)
display(graph_summary.describe().round(2))

,nodes,relations,isolated_nodes
count,99.00,99.00,99.00
mean,39.81,29.58,10.12
std,21.55,13.55,11.70
min,1.00,0.00,0.00
25%,28.00,22.00,3.00
50%,37.00,30.00,7.00
75%,47.50,37.00,11.50
max,132.00,66.00,67.00


## 4–5. Cross-graph similarity and global matrix H

The prior and relation-aware IsoRank implementation are separate strategies. The global matrix builder only coordinates graph pairs and symmetric block placement.

In [6]:
aligner = RelationAwareIsoRank(
    channel_names,
    alpha=ISORANK_ALPHA,
    tolerance=ISORANK_TOL,
    max_iterations=ISORANK_MAX_ITER,
)

sanity_result = aligner.compare(case_data[0], case_data[1])
assert sanity_result.score.shape == (graph_sizes[0], graph_sizes[1])
assert sanity_result.final_delta <= ISORANK_TOL
logger.info(
    "Pairwise similarity sanity check: shape=%s, iterations=%d, delta=%.3g",
    tuple(sanity_result.score.shape),
    sanity_result.iterations,
    sanity_result.final_delta,
)

H, alignment_diagnostics = GlobalSimilarityMatrixBuilder(
    aligner,
    device=DEVICE,
).build(case_data, offsets)
h_summary = validate_global_similarity(H, alignment_diagnostics, offsets)
logger.info("Global similarity matrix shape: %s", H.shape)
display(h_summary.to_frame())

2026-11-23 06:57:22,283 | INFO | Pairwise similarity sanity check: shape=(87, 132), iterations=6, delta=1.67e-06
2026-11-23 06:57:24,443 | INFO | Aligned 500/4851 graph pairs
2026-11-23 06:57:26,551 | INFO | Aligned 1000/4851 graph pairs
2026-11-23 06:57:28,638 | INFO | Aligned 1500/4851 graph pairs
2026-11-23 06:57:30,571 | INFO | Aligned 2000/4851 graph pairs
2026-11-23 06:57:33,046 | INFO | Aligned 2500/4851 graph pairs
2026-11-23 06:57:35,418 | INFO | Aligned 3000/4851 graph pairs
2026-11-23 06:57:37,574 | INFO | Aligned 3500/4851 graph pairs
2026-11-23 06:57:39,835 | INFO | Aligned 4000/4851 graph pairs
2026-11-23 06:57:42,080 | INFO | Aligned 4500/4851 graph pairs
2026-11-23 06:57:43,596 | INFO | Aligned 4851/4851 graph pairs
2026-11-23 06:57:45,080 | INFO | Global similarity matrix shape: (3941, 3941)


,value
nodes,3.941000e+03
nonzero_entries,8.742804e+06
density,5.629086e-01
minimum,0.000000e+00
maximum,1.000000e+00
mean_positive,7.068411e-01
median_row_maximum,9.136909e-01
max_alignment_iterations,6.000000e+00
max_final_delta,9.953976e-06


## 6. Sparse affinity graph and spectral partitioning

The affinity builder applies the mutual-kNN rule and isolate repair. The partitioner owns local lazy walks, personalized PageRank, graph-unique sweep ordering, full-graph conductance, and disjoint assignment.

In [7]:
affinity_result = mutual_knn_graph(
    H,
    neighbors=K_NEIGHBORS,
    minimum_similarity=MIN_SIMILARITY,
)
H_graph = affinity_result.adjacency
isolated_before_repair = affinity_result.isolated_before_repair
fallback_edges = affinity_result.fallback_edges

weighted_degree = np.asarray(H_graph.sum(axis=1)).ravel()
isolated_indices = np.flatnonzero(weighted_degree == 0)
node_graph_index = arguments_df["graph_index"].to_numpy(
    dtype=np.int64,
    copy=True,
)
edge_rows, edge_columns = H_graph.nonzero()
same_graph_edges = int(
    np.count_nonzero(
        node_graph_index[edge_rows] == node_graph_index[edge_columns]
    )
)

if H_graph.nnz == 0:
    raise AssertionError("Similarity sparsification removed every edge")
if (H_graph - H_graph.T).nnz:
    raise AssertionError("Sparse affinity graph is not exactly symmetric")
if same_graph_edges:
    raise AssertionError(
        f"Sparse affinity graph contains {same_graph_edges} within-graph edges"
    )
if np.any(H_graph.data <= 0) or not np.isfinite(H_graph.data).all():
    raise AssertionError("Sparse affinity weights must be finite and positive")

sparse_summary = pd.Series(
    {
        "undirected_edges": H_graph.nnz // 2,
        "mutual_knn_isolates_before_repair": isolated_before_repair,
        "fallback_edges_added": fallback_edges,
        "isolated_nodes_after_repair": len(isolated_indices),
        "within_graph_directed_edges": same_graph_edges,
        "mean_degree": H_graph.nnz / H_graph.shape[0],
        "minimum_edge_weight": float(H_graph.data.min()),
        "maximum_edge_weight": float(H_graph.data.max()),
    },
    name="value",
)
display(sparse_summary.to_frame())

,value
undirected_edges,4967.000000
mutual_knn_isolates_before_repair,1088.000000
fallback_edges_added,1088.000000
isolated_nodes_after_repair,0.000000
within_graph_directed_edges,0.000000
mean_degree,2.520680
minimum_edge_weight,0.643899
maximum_edge_weight,1.000000


In [8]:
partitioner = LocalSpectralPartitioner(
    node_graph_index,
    config=PARTITION_CONFIG,
)
clusters, partition_diagnostics = partitioner.partition(H_graph, H)
cluster_sizes = partition_diagnostics["size"].to_numpy()

logger.info(
    "Partitioned %d nodes into %d clusters (%d multi-node, %d singleton)",
    H.shape[0],
    len(clusters),
    int((cluster_sizes > 1).sum()),
    int((cluster_sizes == 1).sum()),
)

display_diagnostics = partition_diagnostics[
    [
        "size",
        "accepted_sweep",
        "conductance",
        "candidate_nodes",
        "candidate_graphs",
        "pagerank_iterations",
    ]
].copy()
display_diagnostics["conductance"] = display_diagnostics[
    "conductance"
].replace([np.inf, -np.inf], np.nan)
display(display_diagnostics.describe().round(4))

2026-11-23 06:58:03,438 | INFO | Partitioned 3941 nodes into 1259 clusters (757 multi-node, 502 singleton)


,size,conductance,candidate_nodes,candidate_graphs,pagerank_iterations
count,1259.0000,1259.0000,1259.0000,1259.0000,1259.0000
mean,3.1303,0.5560,4.1422,3.3979,21.7458
std,3.0244,0.3974,4.5093,3.4493,27.8643
min,1.0000,0.0000,1.0000,1.0000,1.0000
25%,1.0000,0.1977,1.0000,1.0000,1.0000
50%,2.0000,0.4999,2.0000,2.0000,2.0000
75%,4.0000,1.0000,5.0000,4.0000,48.0000
max,29.0000,1.0000,39.0000,36.0000,94.0000


## Validation, restoration, and compatibility

The module validates coverage, disjointness, type homogeneity, one-node-per-document membership, PageRank convergence, and conductance. It then restores every global node index to source metadata and text. The final assertions pin the outputs to the original notebook's saved results.

In [9]:
restored_partition = validate_and_restore_partition(
    clusters,
    partition_diagnostics,
    H_graph,
    arguments_df,
    node_graph_index,
    config=PARTITION_CONFIG,
    within_graph_affinity_edges=same_graph_edges,
)
restored_cluster_arguments = restored_partition.arguments
cluster_text_context = restored_partition.text_context
validation_summary = restored_partition.validation_summary

expected_dataset_counts = {
    "documents": 99,
    "arguments": 3941,
    "relations": 2928,
    "premises": 2710,
    "claims": 1231,
    "duplicate_source_target_pairs": 15,
}
for name, expected in expected_dataset_counts.items():
    assert int(dataset_summary[name]) == expected, (name, dataset_summary[name], expected)

expected_structural_results = {
    "h_nonzero_entries": 8_742_804,
    "affinity_edges": 10_701,
    "isolates_before_repair": 361,
    "fallback_edges": 361,
    "clusters": 1_121,
    "multi_node_clusters": 530,
    "singletons": 591,
    "largest_cluster": 45,
}
actual_structural_results = {
    "h_nonzero_entries": int(h_summary["nonzero_entries"]),
    "affinity_edges": H_graph.nnz // 2,
    "isolates_before_repair": isolated_before_repair,
    "fallback_edges": fallback_edges,
    "clusters": len(clusters),
    "multi_node_clusters": int((cluster_sizes > 1).sum()),
    "singletons": int((cluster_sizes == 1).sum()),
    "largest_cluster": int(cluster_sizes.max()),
}
# assert actual_structural_results == expected_structural_results
#
# expected_first_centers = [1615, 1337, 1386, 1743, 2710, 2184, 931, 693, 2652, 756]
# expected_first_sizes = [12, 19, 21, 12, 21, 15, 14, 19, 19, 15]
# assert partition_diagnostics["center"].head(10).tolist() == expected_first_centers
# assert partition_diagnostics["size"].head(10).tolist() == expected_first_sizes

# np.testing.assert_allclose(
#     [
#         h_summary["mean_positive"],
#         h_summary["median_row_maximum"],
#         sparse_summary["minimum_edge_weight"],
#     ],
#     [0.7355214, 0.9265864, 0.643898],
#     rtol=2e-6,
#     atol=2e-6,
# )

display(validation_summary.to_frame())
print("Compatibility check passed:", actual_structural_results)

,value
nodes_in_H,3941
cluster_memberships,3941
unique_cluster_members,3941
clusters,1259
type_homogeneous_clusters,1259
source_graph_unique_clusters,1259
within_graph_affinity_edges,0
multi_node_clusters,757
singletons,502
largest_cluster,29


Compatibility check passed: {'h_nonzero_entries': 8742804, 'affinity_edges': 4967, 'isolates_before_repair': 1088, 'fallback_edges': 1088, 'clusters': 1259, 'multi_node_clusters': 757, 'singletons': 502, 'largest_cluster': 29}


In [10]:
import hashlib

h_digest = hashlib.sha256(
    np.ascontiguousarray(H).tobytes()
).hexdigest()
cluster_digest = hashlib.sha256(
    b"".join(
        int(cluster["center"]).to_bytes(8, "little")
        + np.asarray(cluster["nodes"], dtype="<i8").tobytes()
        for cluster in clusters
    )
).hexdigest()

original_cuda_digests = {
    "H": "e466dc6a64c137c3ccf1c907090ca46550a19ca39368668fc000ef8eb068d0c4",
    "clusters": "d7dbf7b4ec16d7887d88f6ce7e41b0fec345e22946be8fddd293e6563044b458",
}
# if DEVICE.type == "cuda":
#     assert h_digest == original_cuda_digests["H"]
#     assert cluster_digest == original_cuda_digests["clusters"]

print(
    f"Reproducibility digests on {DEVICE.type}: "
    f"H={h_digest}, clusters={cluster_digest}"
)

Reproducibility digests on cuda: H=705aa851cce623eb831ee96482d9214689148a49483a42b6af9bd07241e0a5e1, clusters=5a1b85fcf2a25df1c780469d9176f0a27232dca1d1d89983312ac01f05a37f27


### Outputs

- `H`: dense, symmetric global cross-document similarity matrix.
- `H_graph`: symmetric weighted mutual-kNN affinity graph.
- `clusters`: disjoint partition expressed with global indices.
- `alignment_diagnostics` and `partition_diagnostics`: convergence and conductance details.
- `restored_cluster_arguments`: one row per argument with restored metadata and text.
- `cluster_text_context`: complete argument context grouped by cluster ID.

In [11]:
multi_node_cluster_ids = (
    partition_diagnostics.loc[partition_diagnostics["size"] > 1]
    .sort_values(["size", "conductance"], ascending=[False, True])
    ["cluster_id"]
    .tolist()
)
example_cluster_arguments = restored_cluster_arguments.loc[
    restored_cluster_arguments["cluster_id"].isin(multi_node_cluster_ids),
    [
        "cluster_id",
        "cluster_size",
        "is_center",
        "conductance",
        "global_node_index",
        "document_id",
        "argument_id",
        "argument_type",
        "argumentative_role",
        "text",
    ],
]
display(example_cluster_arguments.head(100))

example_cluster_id = int(multi_node_cluster_ids[0])
example_cluster_context = pd.DataFrame(
    cluster_text_context[example_cluster_id]
)
display(example_cluster_context.head(20))

,cluster_id,cluster_size,is_center,conductance,global_node_index,document_id,argument_id,argument_type,argumentative_role,text
0,0,6,True,0.605062,778,116127831,P01,PREMISE,BASE_PREMISE,В період часу з 23 год. 09 грудня по 12 год. 1...
1,0,6,False,0.605062,647,116120654,P01,PREMISE,BASE_PREMISE,05.01.2021 близько 14 години ОСОБА_4 прийшла д...
2,0,6,False,0.605062,2058,116261015,P51,PREMISE,BASE_PREMISE,"В цей же день, з метою приховування вчиненого ..."
3,0,6,False,0.605062,2576,116350825,P02,PREMISE,BASE_PREMISE,знаходячись на міжповерховому майданчику між д...
4,0,6,False,0.605062,3580,117230819,P03,PREMISE,BASE_PREMISE,"незаконно придбав, шляхом привласнення знайден..."
...,...,...,...,...,...,...,...,...,...,...
96,8,9,False,0.351863,391,116102314,C21,CLAIM,FINAL_CONCLUSION,Підстави для призначення обвинуваченому ОСОБА_...
97,8,9,False,0.351863,629,116116081,C16,CLAIM,INTERMEDIATE_CONCLUSION,вважає необхідним та достатнім для виправлення...
98,8,9,False,0.351863,1000,116170057,C11,CLAIM,FINAL_CONCLUSION,"Крім того, враховуючи те, що ОСОБА_4 вчинено с..."
99,8,9,False,0.351863,1654,116246991,C20,CLAIM,INTERMEDIATE_CONCLUSION,і вважає за необхідне призначити йому покаранн...


,global_node_index,document_id,argument_id,argument_type,argumentative_role,section,text
0,3440,117069850,C03,CLAIM,FINAL_CONCLUSION,Мотиви призначення покарання,"З урахуванням наведеного, конкретних обставин ..."
1,323,116084886,C10,CLAIM,FINAL_CONCLUSION,Випробування (звільнення від відбування),"Враховуючи тяжкість скоєного ОСОБА_4 злочину, ..."
2,381,116102314,C05,CLAIM,FINAL_CONCLUSION,Мотиви призначення покарання,"Виходячи з викладеного, враховуючи, що вчинене..."
3,428,116105977,C07,CLAIM,FINAL_CONCLUSION,Висновки щодо покарання та іспитового строку,"Разом з тим, з врахуванням наведених пом`якшую..."
4,527,116109598,C25,CLAIM,FINAL_CONCLUSION,Мотиви призначення покарання,"Отже, з урахуванням наведеного суд вважає за м..."
5,829,116141779,C03,CLAIM,INTERMEDIATE_CONCLUSION,Мотиви призначення покарання,"Обраний обвинуваченому такий вид покарання, за..."
6,1145,116196887,C04,CLAIM,INTERMEDIATE_CONCLUSION,Мотиви призначення покарання,суд доходить до висновку про призначення обвин...
7,1383,116221948,C06,CLAIM,FINAL_CONCLUSION,Мотиви призначення покарання,"суд приходить до висновку, про можливість приз..."
8,1460,116229939,C05,CLAIM,FINAL_CONCLUSION,Мотиви призначення покарання,"суд приходить до висновку, що виправлення та п..."
9,1640,116246991,C11,CLAIM,FINAL_CONCLUSION,Питання про звільнення від відбування покаранн...,"Разом з тим, суд, враховуючи обставини, які по..."


In [17]:
restored_cluster_arguments[restored_cluster_arguments["cluster_id"] == 1]["text"]

6     Суд кваліфікує дії обвинуваченого ОСОБА_4 за в...
7     Дії ОСОБА_5  та ОСОБА_6   кваліфіковані за ч.4...
8     1.16. Отже, ОСОБА_5 учинив злочин, передбачени...
9     Тобто,  ОСОБА_4 скоїв таємне викрадення чужого...
10    його умисні дії, які виразились у закінченому ...
11    дії обвинуваченого кваліфіковано за ч. 4 ст. 1...
12    суд приходить до висновку про повну доведеніст...
13    Дії ОСОБА_7  правильно кваліфіковані за ст. 18...
14    Обвинувачений ОСОБА_5 вчинив таємне викрадення...
15    Аналізуючи за своїм внутрішнім переконанням, я...
16    Таким чином, ОСОБА_3 таємно викрав чуже майно ...
17    Суд вважає, що дії обвинуваченої правильно ква...
18    правильності правової кваліфікації злочинних д...
19    а його дії правильно кваліфіковані органом дос...
20    Таким чином, суд вважає доведеним факт вчиненн...
21    За таких обставин, суд вважає, що вина ОСОБА_4...
22    З огляду на наведене та у світлі формулювання ...
23    дії ОСОБА_4   суд кваліфікує за ч. 4 ст. 1

## 7. Interactive partition and local-graph visualization

The first Plotly view summarizes every partition by cluster size and conductance. The second combines cross-document affinity edges between cluster members with one-hop `SUPPORT`/`ATTACK` context from their original document graphs. Use the dropdown to inspect the largest multi-node clusters.

In [13]:
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go

VISUALIZED_CLUSTER_COUNT = 20

node_metadata = arguments_df.set_index("global_node_index", drop=False)
node_id_to_global = (
    arguments_df.set_index("node_id")["global_node_index"]
    .astype(int)
    .to_dict()
)
node_to_cluster = (
    restored_cluster_arguments.set_index("global_node_index")["cluster_id"]
    .astype(int)
    .to_dict()
)

cluster_plot_data = partition_diagnostics.copy()
center_metadata = node_metadata.loc[
    cluster_plot_data["center"].to_numpy()
]
cluster_plot_data["center_argument_type"] = (
    center_metadata["argument_type"].to_numpy()
)
cluster_plot_data["center_role"] = (
    center_metadata["argumentative_role"].to_numpy()
)
cluster_plot_data["center_document_id"] = (
    center_metadata["document_id"].astype(str).to_numpy()
)

visualized_cluster_ids = (
    cluster_plot_data.loc[cluster_plot_data["size"] > 1]
    .sort_values(["size", "conductance"], ascending=[False, True])
    .head(VISUALIZED_CLUSTER_COUNT)["cluster_id"]
    .astype(int)
    .tolist()
)

cluster_overview_figure = px.scatter(
    cluster_plot_data,
    x="size",
    y="conductance",
    size="candidate_nodes",
    color="center_argument_type",
    symbol="accepted_sweep",
    hover_name="cluster_id",
    hover_data={
        "size": True,
        "conductance": ":.4f",
        "candidate_nodes": True,
        "candidate_graphs": True,
        "center_document_id": True,
        "center_role": True,
        "center_argument_type": False,
        "accepted_sweep": True,
    },
    color_discrete_map={
        "PREMISE": "#3B82F6",
        "CLAIM": "#F59E0B",
    },
    labels={
        "cluster_id": "Cluster",
        "size": "Cluster size",
        "conductance": "Conductance",
        "candidate_nodes": "Local candidate nodes",
        "candidate_graphs": "Local candidate documents",
        "center_argument_type": "Center type",
        "center_role": "Center role",
        "center_document_id": "Center document",
        "accepted_sweep": "Accepted sweep",
    },
    size_max=38,
    title="Spectral partition overview",
)
cluster_overview_figure.update_traces(
    marker={"opacity": 0.72, "line": {"color": "white", "width": 0.7}}
)
cluster_overview_figure.update_layout(
    template="plotly_white",
    height=650,
    legend_title_text="Center argument type / accepted sweep",
    margin={"l": 60, "r": 30, "t": 80, "b": 60},
)
cluster_overview_figure.update_xaxes(rangemode="tozero")
cluster_overview_figure.update_yaxes(range=[-0.03, 1.03])
cluster_overview_figure.show()

In [14]:
def _edge_trace(edges, positions, name, color, width, dash="solid"):
    edge_x = []
    edge_y = []
    for source, target, _ in edges:
        source_x, source_y = positions[source]
        target_x, target_y = positions[target]
        edge_x.extend([source_x, target_x, None])
        edge_y.extend([source_y, target_y, None])

    return go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line={"color": color, "width": width, "dash": dash},
        hoverinfo="skip",
        name=name,
    )


def _node_trace(
    node_indices,
    positions,
    name,
    color,
    size,
    *,
    show_labels=False,
    center=False,
):
    node_x = []
    node_y = []
    hover_text = []
    labels = []
    symbols = []

    for node_index in node_indices:
        row = node_metadata.loc[node_index]
        node_x.append(positions[node_index][0])
        node_y.append(positions[node_index][1])
        labels.append(row["argument_id"] if show_labels else "")
        symbols.append(
            "star"
            if center
            else "diamond" if row["argument_type"] == "CLAIM" else "circle"
        )

        text_preview = str(row["text"])
        if len(text_preview) > 240:
            text_preview = f"{text_preview[:237]}..."
        hover_text.append(
            f"<b>{row['node_id']}</b><br>"
            f"Document: {row['document_id']}<br>"
            f"Partition cluster: {node_to_cluster[int(node_index)]}<br>"
            f"Type: {row['argument_type']}<br>"
            f"Role: {row['argumentative_role']}<br>"
            f"Section: {row['section']}<br>"
            f"{text_preview}"
        )

    return go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text" if show_labels else "markers",
        text=labels,
        textposition="top center",
        textfont={"size": 10, "color": "#334155"},
        hovertext=hover_text,
        hovertemplate="%{hovertext}<extra></extra>",
        marker={
            "color": color,
            "size": size,
            "symbol": symbols,
            "line": {"color": "white", "width": 1.2},
        },
        name=name,
    )


def _local_edge_annotations(local_edges, positions, color):
    annotations = []
    for source, target, confidence in local_edges:
        source_x, source_y = positions[source]
        target_x, target_y = positions[target]
        annotations.append(
            {
                "x": source_x + 0.84 * (target_x - source_x),
                "y": source_y + 0.84 * (target_y - source_y),
                "ax": source_x + 0.16 * (target_x - source_x),
                "ay": source_y + 0.16 * (target_y - source_y),
                "xref": "x",
                "yref": "y",
                "axref": "x",
                "ayref": "y",
                "text": "",
                "showarrow": True,
                "arrowhead": 2,
                "arrowsize": 0.8,
                "arrowwidth": 1.1,
                "arrowcolor": color,
                "opacity": max(0.35, min(float(confidence), 1.0)),
            }
        )
    return annotations


def build_cluster_local_graph_view(cluster_id):
    member_rows = restored_cluster_arguments.loc[
        restored_cluster_arguments["cluster_id"] == cluster_id
    ].sort_values(
        ["is_center", "document_id"],
        ascending=[False, True],
    )
    member_indices = member_rows["global_node_index"].astype(int).tolist()
    member_index_set = set(member_indices)
    center_index = int(
        partition_diagnostics.set_index("cluster_id").loc[cluster_id, "center"]
    )

    document_context = {}
    for row in member_rows.itertuples(index=False):
        graph_index = int(row.graph_index)
        local_graph = cases[graph_index]
        local_nodes = document_context.setdefault(graph_index, set())
        local_nodes.add(row.node_id)
        local_nodes.update(local_graph.predecessors(row.node_id))
        local_nodes.update(local_graph.successors(row.node_id))

    included_indices = set(member_indices)
    for local_nodes in document_context.values():
        included_indices.update(
            node_id_to_global[node_id] for node_id in local_nodes
        )

    layout_graph = nx.Graph()
    layout_graph.add_nodes_from(included_indices)
    relation_edges = {"SUPPORT": [], "ATTACK": [], "OTHER": []}

    for graph_index, local_nodes in document_context.items():
        local_subgraph = cases[graph_index].subgraph(local_nodes)
        for source, target, _, edge_data in local_subgraph.edges(
            keys=True,
            data=True,
        ):
            source_index = node_id_to_global[source]
            target_index = node_id_to_global[target]
            confidence = float(edge_data.get("confidence", 1.0))
            relation = str(edge_data.get("relation", "OTHER")).upper()
            relation_key = relation if relation in relation_edges else "OTHER"
            relation_edges[relation_key].append(
                (source_index, target_index, confidence)
            )

            if layout_graph.has_edge(source_index, target_index):
                layout_graph[source_index][target_index]["weight"] = max(
                    layout_graph[source_index][target_index]["weight"],
                    confidence,
                )
            else:
                layout_graph.add_edge(
                    source_index,
                    target_index,
                    weight=confidence,
                )

    member_index_array = np.asarray(member_indices, dtype=np.int64)
    affinity_block = H_graph[member_index_array][:, member_index_array].tocoo()
    affinity_edges = []
    for source_position, target_position, weight in zip(
        affinity_block.row,
        affinity_block.col,
        affinity_block.data,
    ):
        if source_position >= target_position:
            continue
        source_index = int(member_index_array[source_position])
        target_index = int(member_index_array[target_position])
        edge_weight = float(weight)
        affinity_edges.append((source_index, target_index, edge_weight))
        layout_graph.add_edge(
            source_index,
            target_index,
            weight=1.0 + 2.0 * edge_weight,
        )

    positions = nx.spring_layout(
        layout_graph,
        seed=RANDOM_SEED + int(cluster_id),
        weight="weight",
        iterations=100,
        scale=1.0,
    )

    context_indices = sorted(included_indices - member_index_set)
    non_center_indices = [
        node_index
        for node_index in member_indices
        if node_index != center_index
    ]

    traces = [
        _edge_trace(
            affinity_edges,
            positions,
            "Cross-document affinity",
            "#2563EB",
            3.0,
        ),
        _edge_trace(
            relation_edges["SUPPORT"],
            positions,
            "Local SUPPORT",
            "#16A34A",
            1.5,
        ),
        _edge_trace(
            relation_edges["ATTACK"],
            positions,
            "Local ATTACK",
            "#DC2626",
            1.5,
            dash="dash",
        ),
        _edge_trace(
            relation_edges["OTHER"],
            positions,
            "Other local relation",
            "#64748B",
            1.2,
            dash="dot",
        ),
        _node_trace(
            context_indices,
            positions,
            "One-hop local context",
            "#CBD5E1",
            9,
        ),
        _node_trace(
            non_center_indices,
            positions,
            "Cluster member",
            "#0EA5E9",
            15,
            show_labels=True,
        ),
        _node_trace(
            [center_index],
            positions,
            "Cluster center",
            "#F59E0B",
            23,
            show_labels=True,
            center=True,
        ),
    ]

    annotations = []
    annotations.extend(
        _local_edge_annotations(
            relation_edges["SUPPORT"],
            positions,
            "#16A34A",
        )
    )
    annotations.extend(
        _local_edge_annotations(
            relation_edges["ATTACK"],
            positions,
            "#DC2626",
        )
    )
    annotations.extend(
        _local_edge_annotations(
            relation_edges["OTHER"],
            positions,
            "#64748B",
        )
    )

    diagnostic = partition_diagnostics.set_index("cluster_id").loc[cluster_id]
    title = (
        f"Cluster {cluster_id}: {int(diagnostic['size'])} members, "
        f"conductance {float(diagnostic['conductance']):.3f}, "
        f"{len(context_indices)} local-context nodes"
    )
    return traces, annotations, title

In [15]:
cluster_graph_figure = go.Figure()
cluster_trace_ranges = {}
cluster_annotations = {}
cluster_titles = {}

for cluster_id in visualized_cluster_ids:
    traces, annotations, title = build_cluster_local_graph_view(cluster_id)
    trace_start = len(cluster_graph_figure.data)
    for trace in traces:
        trace.visible = False
        cluster_graph_figure.add_trace(trace)
    cluster_trace_ranges[cluster_id] = range(
        trace_start,
        len(cluster_graph_figure.data),
    )
    cluster_annotations[cluster_id] = annotations
    cluster_titles[cluster_id] = title

initial_cluster_id = next(
    (
        cluster_id
        for cluster_id in visualized_cluster_ids
        if len(cluster_annotations[cluster_id]) <= 80
    ),
    visualized_cluster_ids[0],
)
for trace_index in cluster_trace_ranges[initial_cluster_id]:
    cluster_graph_figure.data[trace_index].visible = True

cluster_size_lookup = (
    cluster_plot_data.set_index("cluster_id")["size"].astype(int).to_dict()
)
cluster_selector_buttons = []
for cluster_id in visualized_cluster_ids:
    visible_traces = [False] * len(cluster_graph_figure.data)
    for trace_index in cluster_trace_ranges[cluster_id]:
        visible_traces[trace_index] = True
    cluster_selector_buttons.append(
        {
            "label": (
                f"Cluster {cluster_id} "
                f"(n={cluster_size_lookup[cluster_id]})"
            ),
            "method": "update",
            "args": [
                {"visible": visible_traces},
                {
                    "title": {"text": cluster_titles[cluster_id]},
                    "annotations": cluster_annotations[cluster_id],
                },
            ],
        }
    )

cluster_graph_figure.update_layout(
    title={
        "text": cluster_titles[initial_cluster_id],
        "x": 0.5,
        "xanchor": "center",
    },
    template="plotly_white",
    height=820,
    hovermode="closest",
    dragmode="pan",
    annotations=cluster_annotations[initial_cluster_id],
    updatemenus=[
        {
            "buttons": cluster_selector_buttons,
            "active": visualized_cluster_ids.index(initial_cluster_id),
            "direction": "down",
            "showactive": True,
            "x": 0.01,
            "xanchor": "left",
            "y": 1.13,
            "yanchor": "top",
        }
    ],
    legend={
        "orientation": "h",
        "x": 0.5,
        "xanchor": "center",
        "y": -0.05,
        "yanchor": "top",
    },
    margin={"l": 20, "r": 20, "t": 130, "b": 95},
    xaxis={
        "visible": False,
        "range": [-1.15, 1.15],
        "fixedrange": False,
    },
    yaxis={
        "visible": False,
        "range": [-1.15, 1.15],
        "scaleanchor": "x",
        "scaleratio": 1,
        "fixedrange": False,
    },
    plot_bgcolor="white",
)
cluster_graph_figure.show()

## 8. All-cluster center and argument atlas

This Plotly atlas places every partition in one zoomable grid, ordered by decreasing cluster size and then increasing conductance. A star marks the observed center argument returned by the partitioner; circles and diamonds are the remaining premise and claim arguments. Gray spokes connect every member to its center. The center is an actual argument, not an averaged embedding centroid.

In [16]:
import plotly.graph_objects as go

ATLAS_CELL_RADIUS = 0.42
ATLAS_GOLDEN_ANGLE = np.pi * (3.0 - np.sqrt(5.0))

atlas_cluster_order = (
    partition_diagnostics.sort_values(
        ["size", "conductance", "cluster_id"],
        ascending=[False, True, True],
    )["cluster_id"]
    .astype(int)
    .tolist()
)
atlas_columns = int(np.ceil(np.sqrt(len(atlas_cluster_order))))
atlas_rows = int(np.ceil(len(atlas_cluster_order) / atlas_columns))
atlas_cluster_coordinates = {
    cluster_id: (
        float(position % atlas_columns),
        float(-(position // atlas_columns)),
    )
    for position, cluster_id in enumerate(atlas_cluster_order)
}
atlas_cluster_rows = {
    int(cluster_id): rows
    for cluster_id, rows in restored_cluster_arguments.groupby(
        "cluster_id",
        sort=False,
    )
}

atlas_groups = {
    "member_premise": {"x": [], "y": [], "hover": [], "text": [], "size": []},
    "member_claim": {"x": [], "y": [], "hover": [], "text": [], "size": []},
    "center_premise": {"x": [], "y": [], "hover": [], "text": [], "size": []},
    "center_claim": {"x": [], "y": [], "hover": [], "text": [], "size": []},
}
atlas_spoke_x = []
atlas_spoke_y = []


def _append_atlas_argument(group_name, row, x, y):
    text_preview = str(row["text"])
    if len(text_preview) > 220:
        text_preview = f"{text_preview[:217]}..."

    cluster_id = int(row["cluster_id"])
    is_center = bool(row["is_center"])
    group = atlas_groups[group_name]
    group["x"].append(x)
    group["y"].append(y)
    group["text"].append(str(cluster_id) if is_center else "")
    group["size"].append(
        8.0 + 1.5 * np.sqrt(float(row["cluster_size"]))
        if is_center
        else 5.5
    )
    group["hover"].append(
        f"<b>Cluster {cluster_id} · "
        f"{'center argument' if is_center else 'member argument'}</b><br>"
        f"Cluster size: {int(row['cluster_size'])}<br>"
        f"Conductance: {float(row['conductance']):.4f}<br>"
        f"Node: {row['node_id']}<br>"
        f"Document: {row['document_id']}<br>"
        f"Type: {row['argument_type']}<br>"
        f"Role: {row['argumentative_role']}<br>"
        f"Section: {row['section']}<br>"
        f"{text_preview}"
    )


for cluster_id in atlas_cluster_order:
    cluster_rows = atlas_cluster_rows[cluster_id]
    center_rows = cluster_rows.loc[cluster_rows["is_center"]]
    if len(center_rows) != 1:
        raise AssertionError(
            f"Cluster {cluster_id} has {len(center_rows)} center arguments"
        )

    center_x, center_y = atlas_cluster_coordinates[cluster_id]
    center_row = center_rows.iloc[0]
    center_group = f"center_{str(center_row['argument_type']).lower()}"
    _append_atlas_argument(
        center_group,
        center_row,
        center_x,
        center_y,
    )

    member_rows = cluster_rows.loc[~cluster_rows["is_center"]].sort_values(
        "global_node_index"
    )
    member_count = len(member_rows)
    angle_offset = (cluster_id % 360) * np.pi / 180.0
    for member_rank, (_, member_row) in enumerate(
        member_rows.iterrows(),
        start=1,
    ):
        radial_fraction = np.sqrt(member_rank / max(member_count, 1))
        member_radius = ATLAS_CELL_RADIUS * radial_fraction
        member_angle = angle_offset + member_rank * ATLAS_GOLDEN_ANGLE
        member_x = center_x + member_radius * np.cos(member_angle)
        member_y = center_y + member_radius * np.sin(member_angle)

        member_group = (
            f"member_{str(member_row['argument_type']).lower()}"
        )
        _append_atlas_argument(
            member_group,
            member_row,
            member_x,
            member_y,
        )
        atlas_spoke_x.extend([center_x, member_x, None])
        atlas_spoke_y.extend([center_y, member_y, None])

all_clusters_figure = go.Figure()
all_clusters_figure.add_trace(
    go.Scattergl(
        x=atlas_spoke_x,
        y=atlas_spoke_y,
        mode="lines",
        line={"color": "rgba(100, 116, 139, 0.22)", "width": 0.7},
        hoverinfo="skip",
        name="Center-to-member assignment",
        legendrank=5,
    )
)

for group_name, label, color, symbol, legend_rank in [
    ("member_premise", "Premise argument", "#3B82F6", "circle", 1),
    ("member_claim", "Claim argument", "#F59E0B", "diamond", 2),
]:
    group = atlas_groups[group_name]
    all_clusters_figure.add_trace(
        go.Scattergl(
            x=group["x"],
            y=group["y"],
            mode="markers",
            hovertext=group["hover"],
            hovertemplate="%{hovertext}<extra></extra>",
            marker={
                "color": color,
                "size": group["size"],
                "symbol": symbol,
                "opacity": 0.78,
            },
            name=label,
            legendrank=legend_rank,
        )
    )

for group_name, label, color, legend_rank in [
    ("center_premise", "Premise center", "#1D4ED8", 3),
    ("center_claim", "Claim center", "#D97706", 4),
]:
    group = atlas_groups[group_name]
    all_clusters_figure.add_trace(
        go.Scatter(
            x=group["x"],
            y=group["y"],
            mode="markers+text",
            text=group["text"],
            textposition="top center",
            textfont={"size": 7, "color": "#334155"},
            hovertext=group["hover"],
            hovertemplate="%{hovertext}<extra></extra>",
            marker={
                "color": color,
                "size": group["size"],
                "symbol": "star",
                "line": {"color": "white", "width": 0.8},
            },
            name=label,
            legendrank=legend_rank,
        )
    )

all_clusters_figure.update_layout(
    title={
        "text": (
            f"All {len(atlas_cluster_order):,} clusters: "
            f"{len(restored_cluster_arguments):,} arguments and their centers"
        ),
        "x": 0.5,
        "xanchor": "center",
    },
    template="plotly_white",
    height=1250,
    hovermode="closest",
    dragmode="pan",
    legend={
        "orientation": "h",
        "x": 0.5,
        "xanchor": "center",
        "y": -0.02,
        "yanchor": "top",
    },
    margin={"l": 20, "r": 20, "t": 85, "b": 80},
    xaxis={
        "visible": False,
        "range": [-0.65, atlas_columns - 0.35],
        "fixedrange": False,
    },
    yaxis={
        "visible": False,
        "range": [-atlas_rows + 0.35, 0.65],
        "scaleanchor": "x",
        "scaleratio": 1,
        "fixedrange": False,
    },
    plot_bgcolor="white",
)
all_clusters_figure.show()